### Process ClinVar `variant_summary` into a clinical-label table

This notebook turns NCBI ClinVar's `variant_summary.txt.gz` into a tidy per-amino-acid-substitution
table with a simplified clinical label (`cs_simple`: `p` = Pathogenic/Likely-pathogenic, `b` = Benign/Likely-benign).
That table is the **clinical truth set** consumed by `02_calibrate_functional_data.ipynb`.

It reproduces the logic of the original `process_Clinvar.R` in pandas:
download the newest release → filter (GRCh38, SNV, germline, protein-coding, P/LP or B/LB) →
parse the HGVS protein change into `aapos/aaref/aaalt` → derive a ClinVar `gold_star` review level → write a dated CSV.

At the end we also write a small **LDLR-only** subset that ships with this repo as example data.

In [ ]:
from pathlib import Path
import re
import gzip
import shutil
import urllib.request
from datetime import date

import pandas as pd
import numpy as np


In [ ]:
# Resolve the repo root whether this notebook is run from the repo root or from notebooks/
proj_dir = Path.cwd()
if proj_dir.name == 'notebooks':
    proj_dir = proj_dir.parent

data_dir = proj_dir / 'data'
raw_dir = data_dir / 'raw'            # downloaded ClinVar releases land here
clinvar_dir = data_dir / 'clinvar'    # processed clinical-label tables

raw_dir.mkdir(parents=True, exist_ok=True)
clinvar_dir.mkdir(parents=True, exist_ok=True)


### Download the newest ClinVar release

NCBI serves the current `variant_summary.txt.gz` at the URL below and keeps dated monthly copies under `archive/`.
We download the live file and **date the local filename** by the current year-month so the provenance is explicit.
Set `redownload = False` to reuse a previously downloaded copy. The file is ~100 MB compressed.

In [ ]:
clinvar_url = 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz'
release_tag = date.today().strftime('%Y-%m')   # e.g. '2026-06'
clinvar_gz = raw_dir / f'variant_summary_{release_tag}.txt.gz'

redownload = False
if redownload or not clinvar_gz.exists():
    print(f'Downloading {clinvar_url}\n  -> {clinvar_gz}')
    urllib.request.urlretrieve(clinvar_url, clinvar_gz)
else:
    print(f'Using existing file: {clinvar_gz}')
print('size (MB):', round(clinvar_gz.stat().st_size / 1e6, 1))


### Load and filter

Keep only rows that are usable as a clinical truth set for protein substitutions:
GRCh38 assembly, single-nucleotide variants, germline origin, carrying an HGVS protein change (`(p.…)`),
and a non-conflicting Pathogenic or Benign assertion.

In [ ]:
# variant_summary is a large TSV; only read the columns we need.
usecols = ['Name', 'GeneSymbol', 'Type', 'Assembly', 'OriginSimple',
           'ClinicalSignificance', 'ReviewStatus', 'Chromosome', 'Start', 'LastEvaluated']
df = pd.read_csv(clinvar_gz, sep='\t', usecols=usecols, dtype=str, low_memory=False)
print('raw rows:', len(df))

keep = (
    (df['Assembly'] == 'GRCh38')
    & (df['Type'] == 'single nucleotide variant')
    & df['Name'].str.contains('(p.', regex=False, na=False)
    & (df['OriginSimple'] == 'germline')
    & df['ClinicalSignificance'].str.contains('patho|benign', case=False, na=False)
    & ~df['ClinicalSignificance'].str.contains('conflict', case=False, na=False)
)
df = df.loc[keep].copy()
print('filtered rows:', len(df))


### Parse the HGVS name into refseq / coding change / protein change

ClinVar `Name` looks like `NM_000527.5(LDLR):c.1A>T (p.Met1Leu)`.
The regex captures the RefSeq id, gene, coding position/ref/alt, and the protein ref-AA / position / alt-AA.
Three-letter amino acids are mapped to single letters; `Ter` → `*` and a synonymous `=` keeps the reference AA.

In [ ]:
AA3_TO_1 = {
    'Ala':'A','Arg':'R','Asn':'N','Asp':'D','Cys':'C','Gln':'Q','Glu':'E','Gly':'G',
    'His':'H','Ile':'I','Leu':'L','Lys':'K','Met':'M','Phe':'F','Pro':'P','Ser':'S',
    'Thr':'T','Trp':'W','Tyr':'Y','Val':'V','Ter':'*',
}

# groups: 1=refseq 2=gene 3=cds_pos 4=ref 5=alt 6=aaref3 7=aapos 8=aaalt3
pat = re.compile(r'(.*)\((.*)\):c\.(\d+)(\D)>(\D) \(p\.(\D*)(\d+)(\D*)\)')
m = df['Name'].str.extract(pat)
m.columns = ['ncbi_refseq', 'gene_paren', 'cds_pos', 'ref', 'alt', 'aaref3', 'aapos', 'aaalt3']

def to_one(aa3, fallback=None):
    if aa3 == '=':           # synonymous: alt AA equals ref AA
        return fallback
    return AA3_TO_1.get(aa3, np.nan)

aaref = m['aaref3'].map(lambda x: AA3_TO_1.get(x, np.nan))
aaalt = [to_one(alt3, ref1) for alt3, ref1 in zip(m['aaalt3'], aaref)]


### Derive ClinVar review level (`gold_star`)

Maps the free-text `ReviewStatus` to ClinVar's 0–4 gold-star scale so downstream analyses can
filter by assertion confidence.

In [ ]:
def gold_star(status):
    s = str(status)
    if 'practice guideline' in s:
        return 4
    if 'reviewed by expert panel' in s:
        return 3
    if 'criteria provided, multiple submitters, no conflicts' in s:
        return 2
    if ('criteria provided, single submitter' in s
            or 'criteria provided, conflicting interpretations' in s):
        return 1
    return 0

df['gold_star'] = df['ReviewStatus'].map(gold_star)


### Assemble the output table and the simplified label `cs_simple`

`cs_simple` collapses `ClinicalSignificance` to `p` (anything Pathogenic) or `b` (anything Benign).
Rows whose protein change could not be parsed are dropped.

In [ ]:
out = pd.DataFrame({
    'gene': df['GeneSymbol'].values,
    'ncbi_refseq': m['ncbi_refseq'].values,
    'aapos': pd.to_numeric(m['aapos'], errors='coerce').values,
    'aaref': aaref.values,
    'aaalt': aaalt,
    'cds_pos': pd.to_numeric(m['cds_pos'], errors='coerce').values,
    'ref': m['ref'].values,
    'alt': m['alt'].values,
    'chrom': df['Chromosome'].values,
    'genomic_pos': df['Start'].values,
    'ClinicalSignificance': df['ClinicalSignificance'].values,
    'gold_star': df['gold_star'].values,
    'LastEvaluated': pd.to_datetime(df['LastEvaluated'], errors='coerce').dt.date.values,
})

out['cs_simple'] = np.where(
    out['ClinicalSignificance'].str.contains('patho', case=False, na=False), 'p',
    np.where(out['ClinicalSignificance'].str.contains('benign', case=False, na=False), 'b', pd.NA))

out = out.dropna(subset=['aaref', 'aapos', 'aaalt']).reset_index(drop=True)
print('final rows:', len(out))
out.head()


### Write the dated, genome-wide table

This is the full processed truth set. Notebook 2 reads a table in exactly this schema.

In [ ]:
full_out = clinvar_dir / f'clinvar_coding_GRCh38_{release_tag}.csv.gz'
out.to_csv(full_out, index=False)
print('wrote', full_out, f'({len(out):,} rows)')


### Write the example subset (LDLR only)

To keep the repo lightweight, the bundled example data is just the LDLR rows.
Swap `example_gene` for your own gene, or skip this cell and use the full table above.

In [ ]:
example_gene = 'LDLR'
sub = out.loc[out['gene'] == example_gene].copy()
sub_out = clinvar_dir / f'clinvar_coding_GRCh38_{release_tag}_{example_gene}.csv.gz'
sub.to_csv(sub_out, index=False)
print('wrote', sub_out, f'({len(sub):,} rows)')
print(sub['cs_simple'].value_counts(dropna=False))
